In [1]:
pip install pandas xarray netCDF4 openpyxl

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install h5py

Note: you may need to restart the kernel to use updated packages.


In [3]:
import numpy as np
#import gplately.pygplates as pygplates
#import ptt
import gplately
import gplately.tools as tools
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader as shpreader
import netCDF4
import warnings
from scipy import ndimage
import glob, os
from rasterio.features import rasterize
from rasterio.transform import from_bounds
import matplotlib.gridspec as gridspec
#from slabdip import SlabDipper
import matplotlib
import pandas as pd
#from cmcrameri import cm
#import gplately.pygplates as pygplates
import glob, os
import matplotlib.pyplot as plt
from plate_model_manager import PlateModelManager

from gplately import pygplates

import geopandas as gpd

# matplotlib.rcParams['font.family'] = 'Arial'
# matplotlib.rcParams['pdf.fonttype'] = 42
# matplotlib.rcParams.update({
#     "pdf.use14corefonts": True
# })

reconstruction_times = np.arange(0,1800,1)
extent_global = [-180,180,-90,90]

2026-04-22 11:13:27 - gplately - ERROR - Failed to import PyGMT. PyGMT requires Python>=3.11.
2026-04-22 11:13:28 - gplately - ERROR - Failed to import PyGMT. PyGMT requires Python>=3.11.


# Import 1.8Ga_model_optimised_mantle_ref_frame_20240725

In [4]:
import pygplates
import glob
import os

use_local_files = True

if use_local_files:
    # Method 1: manually point to files
    #input_directory = 'shirmard25'
    #input_directory = r'./shirmard25'  # Raw string to avoid issues with backslashes
    input_directory_rot = r'C:\Users\hshi9735\OneDrive - The University of Sydney (Students)\PhD_1_Notebooks\Deposits_Rev2\Muller25_1.8_Ga_OMR_NNR_TPW_ref_frame\optimisation'
    input_directory = r'C:\Users\hshi9735\OneDrive - The University of Sydney (Students)\PhD_1_Notebooks\Deposits_Rev2\Muller25_1.8_Ga_OMR_NNR_TPW_ref_frame'
    #input_directory_nc = r'C:\Users\hshi9735\OneDrive - The University of Sydney (Students)\PhD_1_Notebooks\Deposits_Rev2\Muller25_1.8_Ga_OMR_NNR_TPW_ref_frame\Cao2024-Optimised-COB-Masks'
    # Load both rotation files
    rotation_filenames = glob.glob(os.path.join(input_directory_rot, '*.rot')) # Include all rot files
    rotation_model = pygplates.RotationModel(rotation_filenames)

    # Load static polygons
    static_polygons = pygplates.FeatureCollection(os.path.join(input_directory, 'static_polygons.gpmlz'))
    COBs = os.path.join(input_directory, 'COBfile_1800_0.gpml')  # Use updated COB file

    # Load topology files and combine them into a single FeatureCollection
    topology_filenames = [
        os.path.join(input_directory, '250-0_plate_boundaries.gpml'),
        os.path.join(input_directory, '410-250_plate_boundaries.gpml'),
        os.path.join(input_directory, '1800-1000_plate_boundaries.gpml'),
        os.path.join(input_directory, '1000-410-plate-boundaries.gpml'),  # Corrected file name
        os.path.join(input_directory, '1000-410-Convergence.gpml'),
        os.path.join(input_directory, '1000-410-Divergence.gpml'),
        os.path.join(input_directory, '1000-410-Transforms.gpml'),
        os.path.join(input_directory, 'TopologyBuildingBlocks.gpml'),
    ]

    # Create an empty list to hold all the features
    all_topology_features = []

    # Load each file and append its features to the list
    for topology_filename in topology_filenames:
        fc = pygplates.FeatureCollection(topology_filename)
        all_topology_features.extend(fc)

    # Create a single FeatureCollection from the combined features
    topology_features = pygplates.FeatureCollection(all_topology_features)

    # Optionally load other geological features if needed
    #continent_contour_features = pygplates.FeatureCollection(os.path.join(input_directory_nc, 'continent_contour_features.gpmlz'))
    continents = pygplates.FeatureCollection(os.path.join(input_directory, 'shapes_continents.gpmlz'))

    COBs = pygplates.FeatureCollection(os.path.join(input_directory, 'COBfile_1800_0.gpml'))
    shapes_coasts = pygplates.FeatureCollection(os.path.join(input_directory, 'shapes_coasts.gpmlz'))

    # Create PlateReconstruction model
    
    # model = gplately.PlateReconstruction(rotation_model=rotation_model, 
    #                                      topology_features=topology_features, 
    #                                      static_polygons=static_polygons)

    model = gplately.PlateReconstruction(rotation_model=rotation_model,
                                         anchor_plate_id= 14, 
                                         topology_features=topology_features, 
                                         static_polygons=static_polygons)


    # Now you can proceed with your reconstruction and plotting...


# Load Random points

In [5]:
# shp

In [6]:
# import geopandas as gpd
# import pandas as pd
# import gplately

# # Read the shapefile into a GeoDataFrame
# gdf = gpd.read_file('./Inputs/Deposits_with_craton_parameters.shp')  # Replace with the actual path to your shapefile

# # Rename the columns as requested
# gdf = gdf.rename(columns={
#     'TM2': 'weight',
#     'Lat_': 'present_lat',
#     'Lon_': 'present_lon',
#     'Age': 'age (Ma)',
#     'Type' : 'Type'
# })

# # Dictionary to hold DataFrames for each type
# metal_dict = dict()

# # List of commodities (not used in the filtering, just for reference)
# commodities = ['Cu (Mt)', 'Pb (Mt)', 'Zn (Mt)', 'Ni (Mt)']

# # Get unique types from the 'Type' column
# types = gdf['Type'].unique()

# # Filter data by type
# for type_name in types:
#     df = gdf[gdf['Type'] == type_name]
    
#     # Ensure that the 'age (Ma)' column is present and filter data
#     if 'age (Ma)' in df.columns:
#         df = df[df['age (Ma)'].notna()]
#         df = df[df['age (Ma)'] > 0]  # Remove rows where age = 0
#         df = df[df['age (Ma)'] <= 1800]
        
#         if df.shape[0] > 0:
#             metal_dict[type_name] = df

# # Process the filtered data
# pts_dict = dict()
# for i, t in enumerate(types):
#     df = metal_dict[t]
    
#     # Extract Lon/Lat from the geometry column and create points for gplately
#     pts_dict[t] = gplately.Points(model, df['geometry'].x, df['geometry'].y)

# # You can now proceed with the rest of your code (e.g., plotting symbols)
# symbols = ['o', 'v', 's', '*', 'd', '^', 'P']*2


<!-- ### Create random continental locations

This is not really being used currently. We have to think more on how to compare our statistical correlations to a random baseline. -->

In [7]:
# import pandas as pd
# import gplately

# # Read the CSV file into a DataFrame
# df = pd.read_csv('./Inputs/backarc_coords_0.5.csv')  # Replace with the actual path to your CSV file

# # Filter out rows where 'age (Ma)' is NaN, 0, or > 1800
# df = df[df['age (Ma)'].notna()]
# df = df[df['age (Ma)'] > 0]
# df = df[df['age (Ma)'] <= 1800]

# # Create a dictionary of points using 'present_lon' and 'present_lat'
# pts_dict = {idx: gplately.Points(model, row['present_lon'], row['present_lat']) for idx, row in df.iterrows()}

# # Continue with plotting or further analysis if needed
# symbols = ['o', 'v', 's', '*', 'd', '^', 'P'] * 2


In [8]:
# parallelize 

In [9]:
# print(df.columns)

# Load craton database

In [10]:
reconstruction_time_map = 50

In [11]:
### Load craton database
Craton_HS_filename = "Inputs/merged_segmented_craton.shp"
Craton_HS_features = pygplates.FeatureCollection(Craton_HS_filename)
reconstructed_Craton_HS_features = model.reconstruct(Craton_HS_features, reconstruction_time_map)
#deposits = '/data/Craton_HS_2B_PLATEID1/Craton_Deposits_PLATEID1.gpml'

In [12]:
from scipy.spatial import cKDTree

def KD_dist(lons0, lats0, lons1, lats1, k=1):
    """ Function to efficiently query nearest-neighbour distances """
    x0, y0, z0 = gplately.tools.lonlat2xyz(lons0, lats0)
    x1, y1, z1 = gplately.tools.lonlat2xyz(lons1, lats1)
    
    tree = cKDTree(np.c_[x0, y0, z0])
    d, idx = tree.query(np.c_[x1, y1, z1], k=k)
    return d*gplately.EARTH_RADIUS, idx

In [13]:
# # Initialize unique_times as a set to collect unique ages
# unique_times = set()

# # Iterate through each row in the DataFrame to collect unique ages
# unique_times = unique_times.union(set(df['age (Ma)'].to_numpy(dtype=int)))

# # You no longer need 'metal_dict' because you are working directly with 'df'
# # If you need to store or process the filtered data, you can continue working with 'df'


# Merge (1,2,3,4,5) and Extract All Features

In [14]:
# Parallel

In [15]:
# identified several major performance bottlenecks. Here's an optimized version that should run significantly faster:

In [16]:
# import pandas as pd
# import numpy as np
# from scipy.spatial import cKDTree
# import gplately
# from joblib import Parallel, delayed
# import gc

# # ==================== STEP 1: Read CSV and Prepare Data ====================
# print("Reading CSV file...")
# df = pd.read_csv('./Inputs/backarc_coords.csv')
# print(f"Initial rows: {len(df)}")

# # Filter out rows where 'age (Ma)' is NaN, 0, or > 1800
# df = df[df['age (Ma)'].notna()]
# df = df[df['age (Ma)'] > 0]
# df = df[df['age (Ma)'] <= 1800]

# # IMPORTANT: Reset index to ensure continuous indexing from 0
# df = df.reset_index(drop=True)

# print(f"Filtered rows: {len(df)}")

# # Extract coordinates for the processing code
# pts_lons = df['present_lon'].values
# pts_lats = df['present_lat'].values

# print(f"Coordinates extracted:")
# print(f"  - Longitude range: [{pts_lons.min():.2f}, {pts_lons.max():.2f}]")
# print(f"  - Latitude range: [{pts_lats.min():.2f}, {pts_lats.max():.2f}]")
# print(f"  - Total points: {len(pts_lons)}")

# # ==================== STEP 2: Configuration ====================
# # Configuration parameters
# a = 20  # Years for median calculations
# g = 20  # Years for gradient calculations

# # Get unique times from the age column and create age-to-indices mapping
# unique_times = sorted(df['age (Ma)'].astype(int).unique())
# print(f"\nFound {len(unique_times)} unique time steps to process")

# # Create a mapping of time -> indices for fast lookup
# time_to_indices = {}
# for time in unique_times:
#     time_to_indices[time] = df[df['age (Ma)'].astype(int) == time].index.values

# print("Created time-to-indices mapping")

# # ==================== STEP 3: Define Functions ====================
# def KD_dist(x1, y1, x2, y2):
#     """Calculate KDTree distance between two sets of coordinates."""
#     xyz1 = np.c_[gplately.tools.lonlat2xyz(x1, y1)]
#     xyz2 = np.c_[gplately.tools.lonlat2xyz(x2, y2)]
#     tree = cKDTree(xyz1)
#     distances, neighbours = tree.query(xyz2)
#     return distances * 6371, neighbours  # Convert to kilometers


# def process_time_step(time, time_indices, pts_lons_subset, pts_lats_subset, model, Craton_HS_features, a, g):
#     """Process a single time step and return results for that time."""
    
#     # Get indices for this time step
#     index_deposit_age = time_indices
    
#     if len(index_deposit_age) == 0:
#         return None
    
#     # Create Points object ONLY for deposits at this time (much smaller!)
#     pts = gplately.Points(model, pts_lons_subset, pts_lats_subset)
    
#     # Reconstruct craton features for the current time
#     reconstructed_craton_features = model.reconstruct(Craton_HS_features, time)
    
#     # Calculate craton centroids ONCE
#     craton_centroids = np.empty((len(reconstructed_craton_features), 2))
#     for f, feature in enumerate(reconstructed_craton_features):
#         geometry = feature.get_reconstructed_geometry()
#         craton_centroids[f, :] = geometry.get_centroid().to_lat_lon()

#     # Convert craton centroids to xyz coordinates for KDTree search ONCE
#     cx, cy, cz = gplately.tools.lonlat2xyz(craton_centroids[:, 1], craton_centroids[:, 0])
#     tree_craton = cKDTree(np.c_[cx, cy, cz])

#     # Tessellate subduction zones for the current time ONCE
#     subduction_data = model.tessellate_subduction_zones(time, np.deg2rad(0.5), ignore_warnings=True)
#     subduction_lon = subduction_data[:, 0]
#     subduction_lat = subduction_data[:, 1]
#     subduction_convergence = np.fabs(subduction_data[:, 2]) * 1e-2 * np.cos(np.radians(subduction_data[:, 3]))
#     subduction_migration = np.fabs(subduction_data[:, 4]) * 1e-2 * np.cos(np.radians(subduction_data[:, 5]))
#     subduction_norm = subduction_data[:, 7]

#     # Convert lon/lat to xyz coordinates for subduction zones ONCE
#     sx, sy, sz = gplately.tools.lonlat2xyz(subduction_lon, subduction_lat)
#     tree_sz = cKDTree(np.c_[sx, sy, sz])

#     # Reconstruct points for current time (only the subset)
#     rlons, rlats = pts.reconstruct(time, return_array=True)

#     # Apply strict validation for latitude and longitude values
#     valid_mask = (rlats >= -90) & (rlats <= 90) & (rlons >= -180) & (rlons <= 180)
#     rlons = rlons[valid_mask]
#     rlats = rlats[valid_mask]

#     if len(rlats) == 0 or len(rlons) == 0:
#         return None

#     # Calculate deposit velocities at time of formation
#     deposit_velocity_components = model.get_point_velocities(rlons, rlats, time)
#     deposit_velocity_magnitude = np.hypot(deposit_velocity_components[:, 0], deposit_velocity_components[:, 1])

#     # Find closest craton to the deposits
#     distance_to_craton, neighbours = KD_dist(
#         craton_centroids[:, 1], craton_centroids[:, 0], rlons, rlats
#     )

#     # ==================== INSTANTANEOUS VALUES ====================
#     instantaneous_results = {}
    
#     # Pre-calculate all craton geometries to avoid repeated calls
#     craton_geometries = [feature.get_reconstructed_geometry() for feature in reconstructed_craton_features]
    
#     for loop_idx, (d2r, neighbour, df_index) in enumerate(zip(distance_to_craton, neighbours, index_deposit_age)):
#         if not valid_mask[loop_idx]:
#             continue
            
#         geometry = craton_geometries[neighbour]

#         # Get craton properties
#         craton_coordinates = geometry.to_lat_lon_array()

#         # Calculate craton azimuth
#         yx_distances = craton_coordinates[1:] - craton_coordinates[:-1]
#         craton_angles = np.rad2deg(np.arctan2(yx_distances[:, 1], yx_distances[:, 0]))
#         craton_angles[craton_angles < 0] += 180

#         # Calculate craton velocities and azimuth
#         craton_velocity_components = model.get_point_velocities(craton_coordinates[:, 1], craton_coordinates[:, 0], time)
#         craton_velocity_magnitude = np.hypot(craton_velocity_components[:, 0], craton_velocity_components[:, 1])

#         # Find nearest subduction zone segment
#         rx, ry, rz = gplately.tools.lonlat2xyz(craton_coordinates[:, 1], craton_coordinates[:, 0])
#         d2s, sz_neighbours = tree_sz.query(np.c_[rx, ry, rz])
#         d2s_mean = d2s.mean() * 6371

#         # Calculate subduction properties
#         sz_convergence = subduction_convergence[sz_neighbours].mean()
#         sz_migration = subduction_migration[sz_neighbours].mean()
#         sz_norm_vals = subduction_norm[sz_neighbours].copy()
#         sz_norm_vals[sz_norm_vals > 180] -= 180

#         # Calculate obliquity
#         sz_obliquity = abs(sz_norm_vals.mean() - craton_angles.mean()) % 180
#         if sz_obliquity > 90:
#             sz_obliquity = 180 - sz_obliquity

#         # Store instantaneous values as LIST for faster processing
#         instantaneous_results[df_index] = [
#             craton_velocity_magnitude.mean(),  # cr_ve
#             deposit_velocity_magnitude[loop_idx],  # dp_ve
#             d2s_mean,  # ds_cr_tr
#             sz_convergence,  # sb_ve
#             sz_migration,  # tr_adv
#             sz_obliquity,  # tr_cr_an
#             rlats[loop_idx],  # lat
#             rlons[loop_idx]  # lon
#         ]

#     # ==================== MEDIAN VALUES (over 'a' years) ====================
#     all_deposit_vels = []
#     all_craton_vels = []
#     all_craton_azimuths = []
#     all_obliquities = []
#     all_convergences = []
#     all_migrations = []
    
#     for year_offset in range(a):
#         avg_time = time - year_offset
#         rlons_temp, rlats_temp = pts.reconstruct(avg_time, return_array=True)

#         valid_mask_temp = (rlats_temp >= -90) & (rlats_temp <= 90) & (rlons_temp >= -180) & (rlons_temp <= 180)
#         rlons_temp = rlons_temp[valid_mask_temp]
#         rlats_temp = rlats_temp[valid_mask_temp]

#         if len(rlons_temp) == 0:
#             continue

#         # Calculate deposit velocities
#         deposit_velocity_components_temp = model.get_point_velocities(rlons_temp, rlats_temp, avg_time)
#         deposit_velocity_magnitude_temp = np.hypot(deposit_velocity_components_temp[:, 0], deposit_velocity_components_temp[:, 1])
#         all_deposit_vels.append(deposit_velocity_magnitude_temp)

#         # Convert to xyz ONCE
#         deposit_xyz = np.c_[gplately.tools.lonlat2xyz(rlons_temp, rlats_temp)]

#         # Find nearest subduction and craton
#         d2sz, sz_neighbours_temp = tree_sz.query(deposit_xyz)
#         d2craton, craton_neighbours_temp = tree_craton.query(deposit_xyz)

#         all_convergences.append(subduction_convergence[sz_neighbours_temp].mean())
#         all_migrations.append(subduction_migration[sz_neighbours_temp].mean())

#         # Calculate craton velocities
#         craton_velocity_components_temp = model.get_point_velocities(
#             craton_centroids[craton_neighbours_temp, 1], craton_centroids[craton_neighbours_temp, 0], avg_time
#         )
#         craton_velocity_magnitude_temp = np.hypot(craton_velocity_components_temp[:, 0], craton_velocity_components_temp[:, 1])
#         craton_velocity_angle_temp = np.rad2deg(np.arctan2(craton_velocity_components_temp[:, 0], craton_velocity_components_temp[:, 1]))
#         craton_velocity_angle_temp[craton_velocity_angle_temp < 0] += 180

#         all_craton_vels.append(craton_velocity_magnitude_temp)
#         all_craton_azimuths.append(craton_velocity_angle_temp)

#         # Calculate obliquity
#         sz_obliquity_temp = abs(subduction_norm[sz_neighbours_temp].mean() - craton_velocity_angle_temp.mean()) % 180
#         if sz_obliquity_temp > 90:
#             sz_obliquity_temp = 180 - sz_obliquity_temp
#         all_obliquities.append(sz_obliquity_temp)

#     # ==================== GRADIENT VALUES (over 'g' years) ====================
#     grad_deposit_vels = []
#     grad_craton_vels = []
#     grad_craton_azimuths = []
#     grad_obliquities = []
#     grad_convergences = []
#     grad_migrations = []

#     for year_offset in range(g + 1):
#         avg_time = time - year_offset
#         rlons_grad, rlats_grad = pts.reconstruct(avg_time, return_array=True)

#         valid_mask_grad = (rlats_grad >= -90) & (rlats_grad <= 90) & (rlons_grad >= -180) & (rlons_grad <= 180)
#         rlons_grad = rlons_grad[valid_mask_grad]
#         rlats_grad = rlats_grad[valid_mask_grad]

#         if len(rlons_grad) == 0:
#             continue

#         # Calculate deposit velocities
#         deposit_velocity_components_grad = model.get_point_velocities(rlons_grad, rlats_grad, avg_time)
#         deposit_velocity_magnitude_grad = np.hypot(deposit_velocity_components_grad[:, 0], deposit_velocity_components_grad[:, 1])
#         grad_deposit_vels.append(deposit_velocity_magnitude_grad.mean())

#         # Convert to xyz ONCE
#         deposit_xyz_grad = np.c_[gplately.tools.lonlat2xyz(rlons_grad, rlats_grad)]

#         # Find nearest subduction and craton
#         d2sz_grad, sz_neighbours_grad = tree_sz.query(deposit_xyz_grad)
#         d2craton_grad, craton_neighbours_grad = tree_craton.query(deposit_xyz_grad)

#         sz_norm_grad = subduction_norm[sz_neighbours_grad].copy()
#         sz_norm_grad[sz_norm_grad > 180] -= 180

#         # Calculate craton velocities
#         craton_velocity_components_grad = model.get_point_velocities(
#             craton_centroids[craton_neighbours_grad, 1], craton_centroids[craton_neighbours_grad, 0], avg_time
#         )
#         craton_velocity_magnitude_grad = np.hypot(craton_velocity_components_grad[:, 0], craton_velocity_components_grad[:, 1])
#         craton_velocity_angle_grad = np.rad2deg(np.arctan2(craton_velocity_components_grad[:, 0], craton_velocity_components_grad[:, 1]))
#         craton_velocity_angle_grad[craton_velocity_angle_grad < 0] += 180

#         grad_craton_vels.append(craton_velocity_magnitude_grad.mean())
#         grad_craton_azimuths.append(craton_velocity_angle_grad.mean())

#         # Calculate obliquity
#         sz_obliquity_grad = abs(sz_norm_grad.mean() - craton_velocity_angle_grad.mean()) % 180
#         if sz_obliquity_grad > 90:
#             sz_obliquity_grad = 180 - sz_obliquity_grad

#         grad_obliquities.append(sz_obliquity_grad)
#         grad_convergences.append(subduction_convergence[sz_neighbours_grad].mean())
#         grad_migrations.append(subduction_migration[sz_neighbours_grad].mean())

#     # Calculate medians and gradients - return as LIST
#     median_gradient_data = None
    
#     if all_deposit_vels and len(grad_deposit_vels) >= g + 1:
#         median_gradient_data = [
#             np.median(np.concatenate(all_deposit_vels)),  # dp_ve_m
#             np.median(np.concatenate(all_craton_vels)),  # cr_ve_m
#             np.median(np.concatenate(all_craton_azimuths)),  # cr_ve_az_m
#             np.median(all_obliquities),  # tr_cr_an_m
#             np.median(all_convergences),  # sb_ve_m
#             np.median(all_migrations),  # tr_adv_m
#             grad_deposit_vels[g] - grad_deposit_vels[0],  # dp_ve_g
#             grad_craton_vels[g] - grad_craton_vels[0],  # cr_ve_g
#             grad_craton_azimuths[g] - grad_craton_azimuths[0],  # cr_ve_az_c
#             grad_obliquities[g] - grad_obliquities[0],  # tr_cr_an_c
#             grad_convergences[g] - grad_convergences[0],  # sb_ve_g
#             grad_migrations[g] - grad_migrations[0]  # tr_adv_g
#         ]

#     return {
#         'time': time,
#         'instantaneous': instantaneous_results,
#         'median_gradient': median_gradient_data,
#         'indices': index_deposit_age.tolist()
#     }


# # ==================== STEP 4: Run Parallel Processing ====================
# print(f"\nProcessing {len(unique_times)} time steps in parallel on 20 cores...")

# # Prepare data for each time step (only the deposits for that time)
# parallel_tasks = []
# for time in unique_times:
#     indices = time_to_indices[time]
#     pts_lons_subset = pts_lons[indices]
#     pts_lats_subset = pts_lats[indices]
#     parallel_tasks.append((time, indices, pts_lons_subset, pts_lats_subset))

# print(f"Prepared {len(parallel_tasks)} tasks for parallel processing")

# # Run parallel processing
# all_results = Parallel(n_jobs=20, verbose=10)(
#     delayed(process_time_step)(time, indices, lons, lats, model, Craton_HS_features, a, g) 
#     for time, indices, lons, lats in parallel_tasks
# )

# # ==================== STEP 5: FAST MERGE RESULTS ====================
# print("\n" + "="*80)
# print("FAST MERGE: Building data structures...")
# print("="*80)

# # Initialize all columns at once with NaN
# result_columns = ['cr_ve', 'dp_ve', 'ds_cr_tr', 'sb_ve', 'tr_adv', 'tr_cr_an', 'lat', 'lon',
#                  'dp_ve_m', 'cr_ve_m', 'cr_ve_az_m', 'tr_cr_an_m', 'sb_ve_m', 'tr_adv_m',
#                  'dp_ve_g', 'cr_ve_g', 'cr_ve_az_c', 'tr_cr_an_c', 'sb_ve_g', 'tr_adv_g', 'ds_dp_tr']

# for col in result_columns:
#     if col not in df.columns:
#         df[col] = np.nan

# # Define column names for unpacking
# instantaneous_cols = ['cr_ve', 'dp_ve', 'ds_cr_tr', 'sb_ve', 'tr_adv', 'tr_cr_an', 'lat', 'lon']
# median_gradient_cols = ['dp_ve_m', 'cr_ve_m', 'cr_ve_az_m', 'tr_cr_an_m', 'sb_ve_m', 'tr_adv_m',
#                        'dp_ve_g', 'cr_ve_g', 'cr_ve_az_c', 'tr_cr_an_c', 'sb_ve_g', 'tr_adv_g']

# # Collect all data into dictionaries first (FAST)
# print("Collecting data from results...")
# instantaneous_data = {col: {} for col in instantaneous_cols}
# median_gradient_data = {col: {} for col in median_gradient_cols}

# for result_idx, result in enumerate(all_results):
#     if result is None:
#         continue
    
#     if result_idx % 100 == 0:
#         print(f"  Processing result {result_idx}/{len(all_results)}...")
    
#     # Instantaneous values
#     for idx, values in result['instantaneous'].items():
#         for col_idx, col in enumerate(instantaneous_cols):
#             instantaneous_data[col][idx] = values[col_idx]
    
#     # Median and gradient values
#     if result['median_gradient'] is not None:
#         indices = result['indices']
#         for idx in indices:
#             for col_idx, col in enumerate(median_gradient_cols):
#                 median_gradient_data[col][idx] = result['median_gradient'][col_idx]

# # Now update DataFrame columns in BULK (VERY FAST)
# print("Updating DataFrame columns (bulk operation)...")
# for col in instantaneous_cols:
#     if instantaneous_data[col]:
#         indices = list(instantaneous_data[col].keys())
#         values = list(instantaneous_data[col].values())
#         df.loc[indices, col] = values
#         print(f"  Updated {col}: {len(indices)} values")

# for col in median_gradient_cols:
#     if median_gradient_data[col]:
#         indices = list(median_gradient_data[col].keys())
#         values = list(median_gradient_data[col].values())
#         df.loc[indices, col] = values
#         print(f"  Updated {col}: {len(indices)} values")

# print("Results merged successfully!")

# # # ==================== STEP 6: Calculate Final Distances to Trench ====================
# # print("\n" + "="*80)
# # print("Calculating final distances to trench...")
# # print("="*80)

# # unique_times_sorted = sorted(unique_times)

# # # Process final distances in chunks to avoid memory issues
# # chunk_size = 100000
# # num_chunks = (len(df) + chunk_size - 1) // chunk_size

# # print(f"Processing final distances in {num_chunks} chunks...")

# # # Tessellate subduction zones for the last time step ONCE
# # subduction_data_final = model.tessellate_subduction_zones(unique_times_sorted[-1], np.deg2rad(0.5), ignore_warnings=True)
# # subduction_lon_final = subduction_data_final[:, 0]
# # subduction_lat_final = subduction_data_final[:, 1]

# # sx_final, sy_final, sz_final = gplately.tools.lonlat2xyz(subduction_lon_final, subduction_lat_final)
# # tree_sz_final = cKDTree(np.c_[sx_final, sy_final, sz_final])

# # for chunk_idx in range(num_chunks):
# #     start_idx = chunk_idx * chunk_size
# #     end_idx = min((chunk_idx + 1) * chunk_size, len(df))
    
# #     if chunk_idx % 5 == 0:
# #         print(f"  Processing chunk {chunk_idx+1}/{num_chunks}...")
    
# #     # Create Points object for this chunk
# #     pts_chunk = gplately.Points(model, pts_lons[start_idx:end_idx], pts_lats[start_idx:end_idx])
    
# #     # Reconstruct points for final time
# #     rlons_final, rlats_final = pts_chunk.reconstruct(unique_times_sorted[-1], return_array=True)
    
# #     valid_mask_final = (rlats_final >= -90) & (rlats_final <= 90) & (rlons_final >= -180) & (rlons_final <= 180)
# #     rlons_final = rlons_final[valid_mask_final]
# #     rlats_final = rlats_final[valid_mask_final]
    
# #     if len(rlons_final) > 0:
# #         dx_final, dy_final, dz_final = gplately.tools.lonlat2xyz(rlons_final, rlats_final)
# #         d2s_final, _ = tree_sz_final.query(np.c_[dx_final, dy_final, dz_final])
# #         d2s_final = d2s_final * 6371
        
# #         # Assign to the correct indices
# #         valid_indices = np.arange(start_idx, end_idx)[valid_mask_final]
# #         df.loc[valid_indices[:len(d2s_final)], 'ds_dp_tr'] = d2s_final
    
# #     del pts_chunk
# #     gc.collect()

# # # ==================== STEP 7: Save Results ====================
# # print("\n" + "="*80)
# # print("Saving results...")
# # df.to_csv('./Outputs/backarc_data_0.5.csv', index=False)
# # print("Analysis complete! Results saved to ./Outputs/backarc_data_0.5.csv")
# # print("="*80)

In [17]:
# My old buddy!

In [16]:
# Modified-part 1

In [24]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree
import gplately
from joblib import Parallel, delayed
import gc

# ==================== STEP 1: Read CSV and Prepare Data ====================
print("Reading CSV file...")
df = pd.read_csv('./Inputs/coastline_coords.csv')
print(f"Initial rows: {len(df)}")

# Filter out rows where 'age (Ma)' is NaN, 0, or > 1800
df = df[df['age (Ma)'].notna()]
df = df[df['age (Ma)'] > 0]
df = df[df['age (Ma)'] <= 30]

# IMPORTANT: Reset index to ensure continuous indexing from 0
df = df.reset_index(drop=True)

print(f"Filtered rows: {len(df)}")

# Extract coordinates — lon/lat are already reconstructed, use directly
pts_lons = df['lon'].values
pts_lats = df['lat'].values

print(f"Coordinates extracted:")
print(f"  - Longitude range: [{pts_lons.min():.2f}, {pts_lons.max():.2f}]")
print(f"  - Latitude range: [{pts_lats.min():.2f}, {pts_lats.max():.2f}]")
print(f"  - Total points: {len(pts_lons)}")

# ==================== STEP 2: Configuration ====================
# Configuration parameters
a = 20  # Years for median calculations
g = 20  # Years for gradient calculations

# Get unique times from the age column and create age-to-indices mapping
unique_times = sorted(df['age (Ma)'].astype(int).unique())
print(f"\nFound {len(unique_times)} unique time steps to process")

# Create a mapping of time -> indices for fast lookup
time_to_indices = {}
for time in unique_times:
    time_to_indices[time] = df[df['age (Ma)'].astype(int) == time].index.values

print("Created time-to-indices mapping")

# ==================== STEP 3: Define Functions ====================
def KD_dist(x1, y1, x2, y2):
    """Calculate KDTree distance between two sets of coordinates."""
    xyz1 = np.c_[gplately.tools.lonlat2xyz(x1, y1)]
    xyz2 = np.c_[gplately.tools.lonlat2xyz(x2, y2)]
    tree = cKDTree(xyz1)
    distances, neighbours = tree.query(xyz2)
    return distances * 6371, neighbours  # Convert to kilometers


def process_time_step(time, time_indices, rlons_subset, rlats_subset, model, Craton_HS_features, a, g):
    """Process a single time step and return results for that time."""

    index_deposit_age = time_indices

    if len(index_deposit_age) == 0:
        return None

    # Already-reconstructed coordinates for this time step's deposits
    rlons = rlons_subset.copy()
    rlats = rlats_subset.copy()

    # Apply strict validation for latitude and longitude values
    valid_mask = (rlats >= -90) & (rlats <= 90) & (rlons >= -180) & (rlons <= 180)
    rlons = rlons[valid_mask]
    rlats = rlats[valid_mask]

    if len(rlats) == 0 or len(rlons) == 0:
        return None

    # Reconstruct craton features for the current time
    reconstructed_craton_features = model.reconstruct(Craton_HS_features, time)

    # Calculate craton centroids ONCE
    craton_centroids = np.empty((len(reconstructed_craton_features), 2))
    for f, feature in enumerate(reconstructed_craton_features):
        geometry = feature.get_reconstructed_geometry()
        craton_centroids[f, :] = geometry.get_centroid().to_lat_lon()

    # Convert craton centroids to xyz coordinates for KDTree search ONCE
    cx, cy, cz = gplately.tools.lonlat2xyz(craton_centroids[:, 1], craton_centroids[:, 0])
    tree_craton = cKDTree(np.c_[cx, cy, cz])

    # Tessellate subduction zones for the current time ONCE (used for instantaneous values)
    subduction_data = model.tessellate_subduction_zones(time, np.deg2rad(0.5), ignore_warnings=True)
    subduction_lon = subduction_data[:, 0]
    subduction_lat = subduction_data[:, 1]
    subduction_convergence = np.fabs(subduction_data[:, 2]) * 1e-2 * np.cos(np.radians(subduction_data[:, 3]))
    subduction_migration = np.fabs(subduction_data[:, 4]) * 1e-2 * np.cos(np.radians(subduction_data[:, 5]))
    subduction_norm = subduction_data[:, 7]

    # Convert lon/lat to xyz coordinates for subduction zones ONCE
    sx, sy, sz_arr = gplately.tools.lonlat2xyz(subduction_lon, subduction_lat)
    tree_sz = cKDTree(np.c_[sx, sy, sz_arr])

    # Calculate deposit velocities at time of formation
    deposit_velocity_components = model.get_point_velocities(rlons, rlats, time)
    deposit_velocity_magnitude = np.hypot(deposit_velocity_components[:, 0], deposit_velocity_components[:, 1])

    # Find closest craton to the deposits
    distance_to_craton, neighbours = KD_dist(
        craton_centroids[:, 1], craton_centroids[:, 0], rlons, rlats
    )

    # ==================== INSTANTANEOUS VALUES ====================
    instantaneous_results = {}

    # Pre-calculate all craton geometries to avoid repeated calls
    craton_geometries = [feature.get_reconstructed_geometry() for feature in reconstructed_craton_features]

    for loop_idx, (d2r, neighbour, df_index) in enumerate(zip(distance_to_craton, neighbours, index_deposit_age)):
        if not valid_mask[loop_idx]:
            continue

        geometry = craton_geometries[neighbour]

        # Get craton properties
        craton_coordinates = geometry.to_lat_lon_array()

        # Calculate craton azimuth
        yx_distances = craton_coordinates[1:] - craton_coordinates[:-1]
        craton_angles = np.rad2deg(np.arctan2(yx_distances[:, 1], yx_distances[:, 0]))
        craton_angles[craton_angles < 0] += 180

        # Calculate craton velocities and azimuth
        craton_velocity_components = model.get_point_velocities(craton_coordinates[:, 1], craton_coordinates[:, 0], time)
        craton_velocity_magnitude = np.hypot(craton_velocity_components[:, 0], craton_velocity_components[:, 1])

        # Find nearest subduction zone segment
        rx, ry, rz = gplately.tools.lonlat2xyz(craton_coordinates[:, 1], craton_coordinates[:, 0])
        d2s, sz_neighbours = tree_sz.query(np.c_[rx, ry, rz])
        d2s_mean = d2s.mean() * 6371

        # Calculate subduction properties
        sz_convergence = subduction_convergence[sz_neighbours].mean()
        sz_migration = subduction_migration[sz_neighbours].mean()
        sz_norm_vals = subduction_norm[sz_neighbours].copy()
        sz_norm_vals[sz_norm_vals > 180] -= 180

        # Calculate obliquity
        sz_obliquity = abs(sz_norm_vals.mean() - craton_angles.mean()) % 180
        if sz_obliquity > 90:
            sz_obliquity = 180 - sz_obliquity

        # Store instantaneous values as LIST for faster processing
        instantaneous_results[df_index] = [
            craton_velocity_magnitude.mean(),  # cr_ve
            deposit_velocity_magnitude[loop_idx],  # dp_ve
            d2s_mean,                              # ds_cr_tr
            sz_convergence,                        # sb_ve
            sz_migration,                          # tr_adv
            sz_obliquity,                          # tr_cr_an
            rlats[loop_idx],                       # lat
            rlons[loop_idx]                        # lon
        ]

    # Pre-compute deposit xyz (fixed positions, reused across all offsets)
    deposit_xyz = np.c_[gplately.tools.lonlat2xyz(rlons, rlats)]

    # ==================== MEDIAN VALUES (over 'a' years) ====================
    # FIX: Re-tessellate subduction zones at each avg_time so convergence/migration
    #      values actually vary over time (previously reused time's arrays → constant values).
    # FIX: Clamp avg_time >= 0 to avoid negative times returning zero/garbage velocities.
    all_deposit_vels = []
    all_craton_vels = []
    all_craton_azimuths = []
    all_obliquities = []
    all_convergences = []
    all_migrations = []

    for year_offset in range(a):
        avg_time = time - year_offset

        # FIX: Do not go below present day
        if avg_time < 0:
            break

        # FIX: Re-tessellate subduction zones at avg_time
        sub_data_t = model.tessellate_subduction_zones(avg_time, np.deg2rad(0.5), ignore_warnings=True)
        sub_conv_t = np.fabs(sub_data_t[:, 2]) * 1e-2 * np.cos(np.radians(sub_data_t[:, 3]))
        sub_migr_t = np.fabs(sub_data_t[:, 4]) * 1e-2 * np.cos(np.radians(sub_data_t[:, 5]))
        sub_norm_t = sub_data_t[:, 7]
        sx_t, sy_t, sz_t = gplately.tools.lonlat2xyz(sub_data_t[:, 0], sub_data_t[:, 1])
        tree_sz_t = cKDTree(np.c_[sx_t, sy_t, sz_t])

        # Calculate deposit velocities
        deposit_velocity_components_temp = model.get_point_velocities(rlons, rlats, avg_time)
        deposit_velocity_magnitude_temp = np.hypot(
            deposit_velocity_components_temp[:, 0], deposit_velocity_components_temp[:, 1]
        )
        all_deposit_vels.append(deposit_velocity_magnitude_temp)

        # Find nearest subduction and craton using updated tree
        d2sz, sz_neighbours_temp = tree_sz_t.query(deposit_xyz)
        d2craton, craton_neighbours_temp = tree_craton.query(deposit_xyz)

        all_convergences.append(sub_conv_t[sz_neighbours_temp].mean())
        all_migrations.append(sub_migr_t[sz_neighbours_temp].mean())

        # Calculate craton velocities
        craton_velocity_components_temp = model.get_point_velocities(
            craton_centroids[craton_neighbours_temp, 1], craton_centroids[craton_neighbours_temp, 0], avg_time
        )
        craton_velocity_magnitude_temp = np.hypot(
            craton_velocity_components_temp[:, 0], craton_velocity_components_temp[:, 1]
        )
        craton_velocity_angle_temp = np.rad2deg(
            np.arctan2(craton_velocity_components_temp[:, 0], craton_velocity_components_temp[:, 1])
        )
        craton_velocity_angle_temp[craton_velocity_angle_temp < 0] += 180

        all_craton_vels.append(craton_velocity_magnitude_temp)
        all_craton_azimuths.append(craton_velocity_angle_temp)

        # Calculate obliquity using updated sub_norm_t
        sz_obliquity_temp = abs(sub_norm_t[sz_neighbours_temp].mean() - craton_velocity_angle_temp.mean()) % 180
        if sz_obliquity_temp > 90:
            sz_obliquity_temp = 180 - sz_obliquity_temp
        all_obliquities.append(sz_obliquity_temp)

    # ==================== GRADIENT VALUES (over 'g' years) ====================
    # FIX: Re-tessellate subduction zones at each avg_time so gradient differences
    #      are non-trivial (previously grad_convergences[g] - grad_convergences[0]
    #      was always 0 because both indexed into the same static arrays).
    # FIX: Clamp avg_time >= 0 to avoid negative times.
    grad_deposit_vels = []
    grad_craton_vels = []
    grad_craton_azimuths = []
    grad_obliquities = []
    grad_convergences = []
    grad_migrations = []

    for year_offset in range(g + 1):
        avg_time = time - year_offset

        # FIX: Do not go below present day
        if avg_time < 0:
            break

        # FIX: Re-tessellate subduction zones at avg_time
        sub_data_g = model.tessellate_subduction_zones(avg_time, np.deg2rad(0.5), ignore_warnings=True)
        sub_conv_g = np.fabs(sub_data_g[:, 2]) * 1e-2 * np.cos(np.radians(sub_data_g[:, 3]))
        sub_migr_g = np.fabs(sub_data_g[:, 4]) * 1e-2 * np.cos(np.radians(sub_data_g[:, 5]))
        sub_norm_g = sub_data_g[:, 7]
        sx_g, sy_g, sz_g = gplately.tools.lonlat2xyz(sub_data_g[:, 0], sub_data_g[:, 1])
        tree_sz_g = cKDTree(np.c_[sx_g, sy_g, sz_g])

        # Calculate deposit velocities
        deposit_velocity_components_grad = model.get_point_velocities(rlons, rlats, avg_time)
        deposit_velocity_magnitude_grad = np.hypot(
            deposit_velocity_components_grad[:, 0], deposit_velocity_components_grad[:, 1]
        )
        grad_deposit_vels.append(deposit_velocity_magnitude_grad.mean())

        # Find nearest subduction and craton using updated tree
        d2sz_grad, sz_neighbours_grad = tree_sz_g.query(deposit_xyz)
        d2craton_grad, craton_neighbours_grad = tree_craton.query(deposit_xyz)

        sz_norm_grad = sub_norm_g[sz_neighbours_grad].copy()
        sz_norm_grad[sz_norm_grad > 180] -= 180

        # Calculate craton velocities
        craton_velocity_components_grad = model.get_point_velocities(
            craton_centroids[craton_neighbours_grad, 1], craton_centroids[craton_neighbours_grad, 0], avg_time
        )
        craton_velocity_magnitude_grad = np.hypot(
            craton_velocity_components_grad[:, 0], craton_velocity_components_grad[:, 1]
        )
        craton_velocity_angle_grad = np.rad2deg(
            np.arctan2(craton_velocity_components_grad[:, 0], craton_velocity_components_grad[:, 1])
        )
        craton_velocity_angle_grad[craton_velocity_angle_grad < 0] += 180

        grad_craton_vels.append(craton_velocity_magnitude_grad.mean())
        grad_craton_azimuths.append(craton_velocity_angle_grad.mean())

        # Calculate obliquity using updated sub_norm_g
        sz_obliquity_grad = abs(sz_norm_grad.mean() - craton_velocity_angle_grad.mean()) % 180
        if sz_obliquity_grad > 90:
            sz_obliquity_grad = 180 - sz_obliquity_grad

        grad_obliquities.append(sz_obliquity_grad)
        grad_convergences.append(sub_conv_g[sz_neighbours_grad].mean())
        grad_migrations.append(sub_migr_g[sz_neighbours_grad].mean())

    # ==================== CALCULATE MEDIANS AND GRADIENTS ====================
    median_gradient_data = None

    # FIX: Use len(grad_deposit_vels) instead of hardcoded g+1, since the loop
    #      may have broken early due to avg_time < 0 clamping.
    n_grad = len(grad_deposit_vels)

    if all_deposit_vels and n_grad >= 2:
        median_gradient_data = [
            np.median(np.concatenate(all_deposit_vels)),           # dp_ve_m
            np.median(np.concatenate(all_craton_vels)),            # cr_ve_m
            np.median(np.concatenate(all_craton_azimuths)),        # cr_ve_az_m
            np.median(all_obliquities),                            # tr_cr_an_m
            np.median(all_convergences),                           # sb_ve_m
            np.median(all_migrations),                             # tr_adv_m
            grad_deposit_vels[n_grad - 1] - grad_deposit_vels[0], # dp_ve_g
            grad_craton_vels[n_grad - 1] - grad_craton_vels[0],   # cr_ve_g
            grad_craton_azimuths[n_grad - 1] - grad_craton_azimuths[0],  # cr_ve_az_c
            grad_obliquities[n_grad - 1] - grad_obliquities[0],   # tr_cr_an_c
            grad_convergences[n_grad - 1] - grad_convergences[0], # sb_ve_g
            grad_migrations[n_grad - 1] - grad_migrations[0]      # tr_adv_g
        ]

    return {
        'time': time,
        'instantaneous': instantaneous_results,
        'median_gradient': median_gradient_data,
        'indices': index_deposit_age.tolist()
    }


# ==================== STEP 4: Run Parallel Processing ====================
print(f"\nProcessing {len(unique_times)} time steps in parallel on 20 cores...")

# Prepare data for each time step (only the deposits for that time)
parallel_tasks = []
for time in unique_times:
    indices = time_to_indices[time]
    rlons_subset = pts_lons[indices]
    rlats_subset = pts_lats[indices]
    parallel_tasks.append((time, indices, rlons_subset, rlats_subset))

print(f"Prepared {len(parallel_tasks)} tasks for parallel processing")

# Run parallel processing
all_results = Parallel(n_jobs=20, verbose=10)(
    delayed(process_time_step)(time, indices, lons, lats, model, Craton_HS_features, a, g)
    for time, indices, lons, lats in parallel_tasks
)

# ==================== STEP 5: FAST MERGE RESULTS ====================
print("\n" + "="*80)
print("FAST MERGE: Building data structures...")
print("="*80)

# Initialize all columns at once with NaN
result_columns = ['cr_ve', 'dp_ve', 'ds_cr_tr', 'sb_ve', 'tr_adv', 'tr_cr_an', 'lat', 'lon',
                  'dp_ve_m', 'cr_ve_m', 'cr_ve_az_m', 'tr_cr_an_m', 'sb_ve_m', 'tr_adv_m',
                  'dp_ve_g', 'cr_ve_g', 'cr_ve_az_c', 'tr_cr_an_c', 'sb_ve_g', 'tr_adv_g', 'ds_dp_tr']

for col in result_columns:
    if col not in df.columns:
        df[col] = np.nan

# Define column names for unpacking
instantaneous_cols = ['cr_ve', 'dp_ve', 'ds_cr_tr', 'sb_ve', 'tr_adv', 'tr_cr_an', 'lat', 'lon']
median_gradient_cols = ['dp_ve_m', 'cr_ve_m', 'cr_ve_az_m', 'tr_cr_an_m', 'sb_ve_m', 'tr_adv_m',
                        'dp_ve_g', 'cr_ve_g', 'cr_ve_az_c', 'tr_cr_an_c', 'sb_ve_g', 'tr_adv_g']

# Collect all data into dictionaries first (FAST)
print("Collecting data from results...")
instantaneous_data = {col: {} for col in instantaneous_cols}
median_gradient_data_dict = {col: {} for col in median_gradient_cols}

for result_idx, result in enumerate(all_results):
    if result is None:
        continue

    if result_idx % 100 == 0:
        print(f"  Processing result {result_idx}/{len(all_results)}...")

    # Instantaneous values
    for idx, values in result['instantaneous'].items():
        for col_idx, col in enumerate(instantaneous_cols):
            instantaneous_data[col][idx] = values[col_idx]

    # Median and gradient values
    if result['median_gradient'] is not None:
        indices = result['indices']
        for idx in indices:
            for col_idx, col in enumerate(median_gradient_cols):
                median_gradient_data_dict[col][idx] = result['median_gradient'][col_idx]

# Now update DataFrame columns in BULK (VERY FAST)
print("Updating DataFrame columns (bulk operation)...")
for col in instantaneous_cols:
    if instantaneous_data[col]:
        indices = list(instantaneous_data[col].keys())
        values = list(instantaneous_data[col].values())
        df.loc[indices, col] = values
        print(f"  Updated {col}: {len(indices)} values")

for col in median_gradient_cols:
    if median_gradient_data_dict[col]:
        indices = list(median_gradient_data_dict[col].keys())
        values = list(median_gradient_data_dict[col].values())
        df.loc[indices, col] = values
        print(f"  Updated {col}: {len(indices)} values")

print("Results merged successfully!")

Reading CSV file...
Initial rows: 23809378
Filtered rows: 523605
Coordinates extracted:
  - Longitude range: [-165.47, 146.52]
  - Latitude range: [-49.37, 83.08]
  - Total points: 523605

Found 30 unique time steps to process
Created time-to-indices mapping

Processing 30 time steps in parallel on 20 cores...
Prepared 30 tasks for parallel processing


[Parallel(n_jobs=20)]: Using backend LokyBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done   3 out of  30 | elapsed:   47.6s remaining:  7.2min
[Parallel(n_jobs=20)]: Done   7 out of  30 | elapsed:  1.4min remaining:  4.8min
[Parallel(n_jobs=20)]: Done  11 out of  30 | elapsed:  2.1min remaining:  3.6min
[Parallel(n_jobs=20)]: Done  15 out of  30 | elapsed:  2.7min remaining:  2.7min
[Parallel(n_jobs=20)]: Done  19 out of  30 | elapsed:  3.3min remaining:  1.9min
[Parallel(n_jobs=20)]: Done  23 out of  30 | elapsed:  3.8min remaining:  1.1min
[Parallel(n_jobs=20)]: Done  27 out of  30 | elapsed:  4.1min remaining:   27.4s
[Parallel(n_jobs=20)]: Done  30 out of  30 | elapsed:  4.5min finished



FAST MERGE: Building data structures...
  Processing result 0/30...
Updating DataFrame columns (bulk operation)...
  Updated cr_ve: 523605 values
  Updated dp_ve: 523605 values
  Updated ds_cr_tr: 523605 values
  Updated sb_ve: 523605 values
  Updated tr_adv: 523605 values
  Updated tr_cr_an: 523605 values
  Updated lat: 523605 values
  Updated lon: 523605 values
  Updated dp_ve_m: 523605 values
  Updated cr_ve_m: 523605 values
  Updated cr_ve_az_m: 523605 values
  Updated tr_cr_an_m: 523605 values
  Updated sb_ve_m: 523605 values
  Updated tr_adv_m: 523605 values
  Updated dp_ve_g: 523605 values
  Updated cr_ve_g: 523605 values
  Updated cr_ve_az_c: 523605 values
  Updated tr_cr_an_c: 523605 values
  Updated sb_ve_g: 523605 values
  Updated tr_adv_g: 523605 values
Results merged successfully!


In [25]:
df['latabs'] = df['lat'].abs()

In [26]:
# ==================== STEP 6: Calculate Final Distances to Trench ====================
print("\n" + "="*80)
print("Calculating final distances to trench...")
print("="*80)

unique_times_sorted = sorted(unique_times)

# Tessellate subduction zones for the last time step ONCE
subduction_data_final = model.tessellate_subduction_zones(unique_times_sorted[-1], np.deg2rad(0.5), ignore_warnings=True)
subduction_lon_final = subduction_data_final[:, 0]
subduction_lat_final = subduction_data_final[:, 1]

sx_final, sy_final, sz_final = gplately.tools.lonlat2xyz(subduction_lon_final, subduction_lat_final)
tree_sz_final = cKDTree(np.c_[sx_final, sy_final, sz_final])

# Use lon/lat directly — already reconstructed, no chunking needed
rlons_final = df['lon'].values
rlats_final = df['lat'].values

valid_mask_final = (rlats_final >= -90) & (rlats_final <= 90) & (rlons_final >= -180) & (rlons_final <= 180)
rlons_final_valid = rlons_final[valid_mask_final]
rlats_final_valid = rlats_final[valid_mask_final]

print(f"  Valid points: {valid_mask_final.sum()} / {len(df)}")

if len(rlons_final_valid) > 0:
    dx_final, dy_final, dz_final = gplately.tools.lonlat2xyz(rlons_final_valid, rlats_final_valid)
    d2s_final, _ = tree_sz_final.query(np.c_[dx_final, dy_final, dz_final])
    d2s_final = d2s_final * 6371

    valid_indices = np.where(valid_mask_final)[0]
    df.loc[valid_indices, 'ds_dp_tr'] = d2s_final
    print(f"  Assigned ds_dp_tr to {len(valid_indices)} rows")

print("Final distances calculated successfully!")

# ==================== STEP 7: Save Results ====================
print("\n" + "="*80)
print("Saving results...")
df.to_csv('./Outputs/coastline_data.csv', index=False)
print("Analysis complete! Results saved to ./Outputs/coastline_data.csv")
print("="*80)


Calculating final distances to trench...
  Valid points: 523605 / 523605
  Assigned ds_dp_tr to 523605 rows
Final distances calculated successfully!

Saving results...
Analysis complete! Results saved to ./Outputs/coastline_data.csv


In [19]:
# final so far

In [20]:
# import pandas as pd
# import numpy as np
# from scipy.spatial import cKDTree
# import gplately
# from joblib import Parallel, delayed
# import gc

# # ==================== STEP 1: Read CSV and Prepare Data ====================
# print("Reading CSV file...")
# df = pd.read_csv('./Inputs/coastline_coords.csv')
# print(f"Initial rows: {len(df)}")

# df = df[df['age (Ma)'].notna()]
# df = df[df['age (Ma)'] > 0]
# df = df[df['age (Ma)'] <= 1800]
# df = df.reset_index(drop=True)
# print(f"Filtered rows: {len(df)}")

# pts_lons         = df['lon'].values
# pts_lats         = df['lat'].values
# pts_lons_present = df['present_lon'].values
# pts_lats_present = df['present_lat'].values
# pts_ages         = df['age (Ma)'].values

# print(f"Coordinates extracted:")
# print(f"  - Reconstructed lon range: [{pts_lons.min():.2f}, {pts_lons.max():.2f}]")
# print(f"  - Reconstructed lat range: [{pts_lats.min():.2f}, {pts_lats.max():.2f}]")
# print(f"  - Total points: {len(pts_lons)}")

# # ==================== STEP 2: Configuration ====================
# a = 20  # window for median calculations
# g = 20  # window for gradient calculations

# unique_times = sorted(df['age (Ma)'].astype(int).unique())
# max_time     = max(unique_times)

# time_to_indices = {}
# for time in unique_times:
#     time_to_indices[time] = df[df['age (Ma)'].astype(int) == time].index.values

# print(f"\nFound {len(unique_times)} unique time steps")

# # ==================== STEP 3: PRE-COMPUTE ALL TESSELLATIONS ====================
# # Build the full set of times we will ever need:
# #   - every unique deposit time (instantaneous)
# #   - every unique_time - offset for offset in 0..g  (median + gradient)
# # This covers 0 .. max_time+g in steps of 1 Ma.
# all_needed_times = sorted(set(
#     t for base in unique_times for t in range(max(0, base - g), base + 1)
# ))
# print(f"\nPre-computing tessellations for {len(all_needed_times)} time steps...")
# print("This is a one-time cost — each time step will be tessellated exactly once.")

# # --- subduction cache ---
# # sub_cache[t] = dict with arrays: conv, migr, norm, tree_sz
# sub_cache = {}
# for i, t in enumerate(all_needed_times):
#     if i % 50 == 0:
#         print(f"  Tessellating subduction zones: {i}/{len(all_needed_times)} (time={t} Ma)...")
#     sd        = model.tessellate_subduction_zones(t, np.deg2rad(0.5), ignore_warnings=True)
#     conv      = np.fabs(sd[:, 2]) * 1e-2 * np.cos(np.radians(sd[:, 3]))
#     migr      = np.fabs(sd[:, 4]) * 1e-2 * np.cos(np.radians(sd[:, 5]))
#     norm      = sd[:, 7]
#     sx, sy, sz_arr = gplately.tools.lonlat2xyz(sd[:, 0], sd[:, 1])
#     tree      = cKDTree(np.c_[sx, sy, sz_arr])
#     sub_cache[t] = {'conv': conv, 'migr': migr, 'norm': norm, 'tree': tree}
#     del sd, sx, sy, sz_arr

# print("Subduction tessellation cache complete.")

# # --- craton cache ---
# # craton_cache[t] = dict with: centroids (N,2), tree_craton, geometries list
# print(f"\nPre-computing craton reconstructions for {len(all_needed_times)} time steps...")
# craton_cache = {}
# for i, t in enumerate(all_needed_times):
#     if i % 50 == 0:
#         print(f"  Reconstructing cratons: {i}/{len(all_needed_times)} (time={t} Ma)...")
#     features   = model.reconstruct(Craton_HS_features, t)
#     centroids  = np.empty((len(features), 2))
#     geometries = []
#     for f, feat in enumerate(features):
#         geom = feat.get_reconstructed_geometry()
#         centroids[f, :] = geom.get_centroid().to_lat_lon()
#         geometries.append(geom)
#     cx, cy, cz = gplately.tools.lonlat2xyz(centroids[:, 1], centroids[:, 0])
#     tree_cr    = cKDTree(np.c_[cx, cy, cz])
#     craton_cache[t] = {
#         'centroids':   centroids,
#         'tree':        tree_cr,
#         'geometries':  geometries
#     }
#     del features, cx, cy, cz

# print("Craton reconstruction cache complete.")
# print(f"\nCache memory summary:")
# print(f"  sub_cache entries   : {len(sub_cache)}")
# print(f"  craton_cache entries: {len(craton_cache)}")

# # ==================== STEP 4: Helper Functions ====================
# def KD_dist(x1, y1, x2, y2):
#     xyz1 = np.c_[gplately.tools.lonlat2xyz(x1, y1)]
#     xyz2 = np.c_[gplately.tools.lonlat2xyz(x2, y2)]
#     tree = cKDTree(xyz1)
#     distances, neighbours = tree.query(xyz2)
#     return distances * 6371, neighbours


# def interpolate_positions(lons_rec, lats_rec, lons_pres, lats_pres, avg_time, age):
#     """Linear interpolation: frac=0 → present, frac=1 → reconstructed."""
#     if age == 0:
#         return lons_pres.copy(), lats_pres.copy()
#     frac = avg_time / age
#     return (lons_pres + frac * (lons_rec - lons_pres),
#             lats_pres + frac * (lats_rec - lats_pres))


# # ==================== STEP 5: Process One Time Step ====================
# def process_time_step(time, time_indices,
#                       rlons_subset, rlats_subset,
#                       rlons_present_subset, rlats_present_subset,
#                       sub_cache, craton_cache, a, g):

#     index_deposit_age = time_indices
#     if len(index_deposit_age) == 0:
#         return None

#     rlons         = rlons_subset.copy()
#     rlats         = rlats_subset.copy()
#     rlons_present = rlons_present_subset.copy()
#     rlats_present = rlats_present_subset.copy()

#     valid_mask    = (rlats >= -90) & (rlats <= 90) & (rlons >= -180) & (rlons <= 180)
#     rlons         = rlons[valid_mask]
#     rlats         = rlats[valid_mask]
#     rlons_present = rlons_present[valid_mask]
#     rlats_present = rlats_present[valid_mask]

#     if len(rlats) == 0:
#         return None

#     n_deposits = len(rlons)

#     # ---- pull pre-computed data for formation time ----
#     sc  = sub_cache[time]
#     cc  = craton_cache[time]
#     craton_centroids  = cc['centroids']
#     tree_craton       = cc['tree']
#     craton_geometries = cc['geometries']

#     # ==================== INSTANTANEOUS VALUES ====================
#     # Deposit velocities (vectorised — one call for all deposits at this time)
#     dep_vel      = model.get_point_velocities(rlons, rlats, time)
#     dep_vel_mag  = np.hypot(dep_vel[:, 0], dep_vel[:, 1])

#     # Nearest craton to each deposit
#     dist_to_craton, cr_neighbours = KD_dist(
#         craton_centroids[:, 1], craton_centroids[:, 0], rlons, rlats
#     )

#     instantaneous_results = {}

#     for loop_idx, (d2r, neighbour, df_index) in enumerate(
#             zip(dist_to_craton, cr_neighbours, index_deposit_age)):

#         if not valid_mask[loop_idx]:
#             continue

#         geom              = craton_geometries[neighbour]
#         craton_coords     = geom.to_lat_lon_array()

#         yx               = craton_coords[1:] - craton_coords[:-1]
#         craton_angles    = np.rad2deg(np.arctan2(yx[:, 1], yx[:, 0]))
#         craton_angles[craton_angles < 0] += 180

#         # Craton velocities (one call per craton boundary — unavoidable)
#         cr_vel           = model.get_point_velocities(craton_coords[:, 1], craton_coords[:, 0], time)
#         cr_vel_mag       = np.hypot(cr_vel[:, 0], cr_vel[:, 1])

#         rx, ry, rz       = gplately.tools.lonlat2xyz(craton_coords[:, 1], craton_coords[:, 0])
#         d2s, sz_nbrs     = sc['tree'].query(np.c_[rx, ry, rz])
#         d2s_mean         = d2s.mean() * 6371

#         sz_conv          = sc['conv'][sz_nbrs].mean()
#         sz_migr          = sc['migr'][sz_nbrs].mean()
#         sz_norm_v        = sc['norm'][sz_nbrs].copy()
#         sz_norm_v[sz_norm_v > 180] -= 180

#         obliquity        = abs(sz_norm_v.mean() - craton_angles.mean()) % 180
#         if obliquity > 90:
#             obliquity    = 180 - obliquity

#         instantaneous_results[df_index] = [
#             cr_vel_mag.mean(),        # cr_ve
#             dep_vel_mag[loop_idx],    # dp_ve
#             d2s_mean,                 # ds_cr_tr
#             sz_conv,                  # sb_ve
#             sz_migr,                  # tr_adv
#             obliquity,                # tr_cr_an
#             rlats[loop_idx],          # lat
#             rlons[loop_idx]           # lon
#         ]

#     # ==================== COMBINED MEDIAN + GRADIENT LOOP ====================
#     # Single loop covers all offsets 0..g, collecting data for BOTH median and
#     # gradient in one pass — halving the number of velocity/cache lookups.
#     max_offset = max(a, g)  # = g since a == g, but kept general

#     all_dep_vels   = []   # one entry per offset (array over deposits)
#     all_cr_vels    = []
#     all_cr_az      = []
#     all_obliq      = []
#     all_conv       = []
#     all_migr       = []

#     for year_offset in range(max_offset + 1):
#         avg_time = time - year_offset

#         if avg_time < 0:
#             break

#         # Linearly interpolated deposit positions
#         i_lons, i_lats = interpolate_positions(
#             rlons, rlats, rlons_present, rlats_present, avg_time, time
#         )

#         # Pull pre-computed subduction + craton for avg_time
#         sc_t  = sub_cache[avg_time]
#         cc_t  = craton_cache[avg_time]
#         cr_c  = cc_t['centroids']
#         tr_c  = cc_t['tree']

#         dep_xyz = np.c_[gplately.tools.lonlat2xyz(i_lons, i_lats)]

#         # Nearest subduction and craton (pure KDTree lookup — very fast)
#         _, sz_nbrs_t  = sc_t['tree'].query(dep_xyz)
#         _, cr_nbrs_t  = tr_c.query(dep_xyz)

#         # Deposit velocities at interpolated positions
#         dv   = model.get_point_velocities(i_lons, i_lats, avg_time)
#         dv_m = np.hypot(dv[:, 0], dv[:, 1])
#         all_dep_vels.append(dv_m)

#         # Craton velocities
#         cv   = model.get_point_velocities(cr_c[cr_nbrs_t, 1], cr_c[cr_nbrs_t, 0], avg_time)
#         cv_m = np.hypot(cv[:, 0], cv[:, 1])
#         cv_a = np.rad2deg(np.arctan2(cv[:, 0], cv[:, 1]))
#         cv_a[cv_a < 0] += 180

#         all_cr_vels.append(cv_m)
#         all_cr_az.append(cv_a)
#         all_conv.append(sc_t['conv'][sz_nbrs_t].mean())
#         all_migr.append(sc_t['migr'][sz_nbrs_t].mean())

#         obliq = abs(sc_t['norm'][sz_nbrs_t].mean() - cv_a.mean()) % 180
#         if obliq > 90:
#             obliq = 180 - obliq
#         all_obliq.append(obliq)

#     # ---- medians: use first min(a, n_collected) offsets ----
#     n_collected = len(all_dep_vels)
#     n_med       = min(a, n_collected)
#     n_grad      = n_collected  # gradient uses full collected range

#     median_gradient_data = None
#     if n_collected >= 2:
#         med_dep_vels = np.concatenate(all_dep_vels[:n_med])
#         med_cr_vels  = np.concatenate(all_cr_vels[:n_med])
#         med_cr_az    = np.concatenate(all_cr_az[:n_med])

#         median_gradient_data = [
#             np.median(med_dep_vels),                                  # dp_ve_m
#             np.median(med_cr_vels),                                   # cr_ve_m
#             np.median(med_cr_az),                                     # cr_ve_az_m
#             np.median(all_obliq[:n_med]),                             # tr_cr_an_m
#             np.median(all_conv[:n_med]),                              # sb_ve_m
#             np.median(all_migr[:n_med]),                              # tr_adv_m
#             all_dep_vels[n_grad-1].mean() - all_dep_vels[0].mean(),  # dp_ve_g
#             all_cr_vels[n_grad-1].mean()  - all_cr_vels[0].mean(),   # cr_ve_g
#             all_cr_az[n_grad-1].mean()    - all_cr_az[0].mean(),     # cr_ve_az_c
#             all_obliq[n_grad-1]           - all_obliq[0],            # tr_cr_an_c
#             all_conv[n_grad-1]            - all_conv[0],             # sb_ve_g
#             all_migr[n_grad-1]            - all_migr[0]              # tr_adv_g
#         ]

#     return {
#         'time':            time,
#         'instantaneous':   instantaneous_results,
#         'median_gradient': median_gradient_data,
#         'indices':         index_deposit_age.tolist()
#     }


# # ==================== STEP 6: Run Parallel Processing ====================
# print(f"\nProcessing {len(unique_times)} time steps in parallel...")

# parallel_tasks = []
# for time in unique_times:
#     indices              = time_to_indices[time]
#     parallel_tasks.append((
#         time, indices,
#         pts_lons[indices],         pts_lats[indices],
#         pts_lons_present[indices], pts_lats_present[indices]
#     ))

# print(f"Prepared {len(parallel_tasks)} tasks")

# parallel_tasks = parallel_tasks[:60]

# all_results = Parallel(n_jobs=20, verbose=10)(
#     delayed(process_time_step)(
#         time, indices, lons, lats, lons_p, lats_p,
#         sub_cache, craton_cache, a, g
#     )
#     for time, indices, lons, lats, lons_p, lats_p in parallel_tasks
# )

# # ==================== STEP 7: FAST MERGE RESULTS ====================
# print("\n" + "="*80)
# print("FAST MERGE: Building data structures...")
# print("="*80)

# result_columns = [
#     'cr_ve', 'dp_ve', 'ds_cr_tr', 'sb_ve', 'tr_adv', 'tr_cr_an', 'lat', 'lon',
#     'dp_ve_m', 'cr_ve_m', 'cr_ve_az_m', 'tr_cr_an_m', 'sb_ve_m', 'tr_adv_m',
#     'dp_ve_g', 'cr_ve_g', 'cr_ve_az_c', 'tr_cr_an_c', 'sb_ve_g', 'tr_adv_g', 'ds_dp_tr'
# ]
# for col in result_columns:
#     if col not in df.columns:
#         df[col] = np.nan

# instantaneous_cols   = ['cr_ve', 'dp_ve', 'ds_cr_tr', 'sb_ve', 'tr_adv', 'tr_cr_an', 'lat', 'lon']
# median_gradient_cols = [
#     'dp_ve_m', 'cr_ve_m', 'cr_ve_az_m', 'tr_cr_an_m', 'sb_ve_m', 'tr_adv_m',
#     'dp_ve_g', 'cr_ve_g', 'cr_ve_az_c', 'tr_cr_an_c', 'sb_ve_g', 'tr_adv_g'
# ]

# inst_data = {col: {} for col in instantaneous_cols}
# mg_data   = {col: {} for col in median_gradient_cols}

# for result_idx, result in enumerate(all_results):
#     if result is None:
#         continue
#     if result_idx % 100 == 0:
#         print(f"  Merging result {result_idx}/{len(all_results)}...")

#     for idx, values in result['instantaneous'].items():
#         for ci, col in enumerate(instantaneous_cols):
#             inst_data[col][idx] = values[ci]

#     if result['median_gradient'] is not None:
#         for idx in result['indices']:
#             for ci, col in enumerate(median_gradient_cols):
#                 mg_data[col][idx] = result['median_gradient'][ci]

# print("Updating DataFrame (bulk)...")
# for col in instantaneous_cols:
#     if inst_data[col]:
#         df.loc[list(inst_data[col].keys()), col] = list(inst_data[col].values())
#         print(f"  {col}: {len(inst_data[col])} values")

# for col in median_gradient_cols:
#     if mg_data[col]:
#         df.loc[list(mg_data[col].keys()), col] = list(mg_data[col].values())
#         print(f"  {col}: {len(mg_data[col])} values")

# print("\nDone!")

In [21]:
# print(df.head(10).to_string())

In [27]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree
import gplately
from joblib import Parallel, delayed
import gc

# ==================== STEP 1: Read CSV and Prepare Data ====================
print("Reading CSV file...")
df = pd.read_csv('./Inputs/unlabelled_coords.csv')
print(f"Initial rows: {len(df)}")

df = df[df['age (Ma)'].notna()]
df = df[df['age (Ma)'] > 0]
df = df[df['age (Ma)'] <= 1800]
df = df.reset_index(drop=True)
print(f"Filtered rows: {len(df)}")

pts_lons         = df['lon'].values
pts_lats         = df['lat'].values
pts_lons_present = df['present_lon'].values
pts_lats_present = df['present_lat'].values
pts_ages         = df['age (Ma)'].values

print(f"Coordinates extracted:")
print(f"  - Reconstructed lon range: [{pts_lons.min():.2f}, {pts_lons.max():.2f}]")
print(f"  - Reconstructed lat range: [{pts_lats.min():.2f}, {pts_lats.max():.2f}]")
print(f"  - Total points: {len(pts_lons)}")

# ==================== STEP 2: Configuration ====================
a = 20  # window for median (uses time and time-20 only)
g = 20  # window for gradient (uses time and time-20 only)

unique_times = sorted(df['age (Ma)'].astype(int).unique())
max_time     = max(unique_times)

time_to_indices = {}
for time in unique_times:
    time_to_indices[time] = df[df['age (Ma)'].astype(int) == time].index.values

print(f"\nFound {len(unique_times)} unique time steps")

# ==================== STEP 3: PRE-COMPUTE ALL TESSELLATIONS ====================
# Only two time points needed per deposit time: time and time-20 (clamped to 0)
all_needed_times = sorted(set(
    t for base in unique_times
    for t in [base, max(0, base - g)]
))
print(f"\nPre-computing tessellations for {len(all_needed_times)} time steps...")
print("(Only 2 time points per deposit time: t and t-20)")

# --- subduction cache ---
sub_cache = {}
for i, t in enumerate(all_needed_times):
    if i % 100 == 0:
        print(f"  Tessellating subduction zones: {i}/{len(all_needed_times)} (time={t} Ma)...")
    sd        = model.tessellate_subduction_zones(t, np.deg2rad(0.5), ignore_warnings=True)
    conv      = np.fabs(sd[:, 2]) * 1e-2 * np.cos(np.radians(sd[:, 3]))
    migr      = np.fabs(sd[:, 4]) * 1e-2 * np.cos(np.radians(sd[:, 5]))
    norm      = sd[:, 7]
    sx, sy, sz_arr = gplately.tools.lonlat2xyz(sd[:, 0], sd[:, 1])
    tree      = cKDTree(np.c_[sx, sy, sz_arr])
    sub_cache[t] = {'conv': conv, 'migr': migr, 'norm': norm, 'tree': tree}
    del sd, sx, sy, sz_arr

print("Subduction tessellation cache complete.")

# --- craton cache ---
print(f"\nPre-computing craton reconstructions for {len(all_needed_times)} time steps...")
craton_cache = {}
for i, t in enumerate(all_needed_times):
    if i % 100 == 0:
        print(f"  Reconstructing cratons: {i}/{len(all_needed_times)} (time={t} Ma)...")
    features   = model.reconstruct(Craton_HS_features, t)
    centroids  = np.empty((len(features), 2))
    geometries = []
    for f, feat in enumerate(features):
        geom = feat.get_reconstructed_geometry()
        centroids[f, :] = geom.get_centroid().to_lat_lon()
        geometries.append(geom)
    cx, cy, cz = gplately.tools.lonlat2xyz(centroids[:, 1], centroids[:, 0])
    tree_cr    = cKDTree(np.c_[cx, cy, cz])
    craton_cache[t] = {
        'centroids':  centroids,
        'tree':       tree_cr,
        'geometries': geometries
    }
    del features, cx, cy, cz

print("Craton reconstruction cache complete.")
print(f"\nCache summary:")
print(f"  sub_cache entries   : {len(sub_cache)}")
print(f"  craton_cache entries: {len(craton_cache)}")

# ==================== STEP 4: Helper Functions ====================
def KD_dist(x1, y1, x2, y2):
    xyz1 = np.c_[gplately.tools.lonlat2xyz(x1, y1)]
    xyz2 = np.c_[gplately.tools.lonlat2xyz(x2, y2)]
    tree = cKDTree(xyz1)
    distances, neighbours = tree.query(xyz2)
    return distances * 6371, neighbours


def interpolate_positions(lons_rec, lats_rec, lons_pres, lats_pres, avg_time, age):
    """Linear interpolation: frac=0 → present, frac=1 → reconstructed."""
    if age == 0:
        return lons_pres.copy(), lats_pres.copy()
    frac = avg_time / age
    return (lons_pres + frac * (lons_rec - lons_pres),
            lats_pres + frac * (lats_rec - lats_pres))


# ==================== STEP 5: Process One Time Step ====================
def process_time_step(time, time_indices,
                      rlons_subset, rlats_subset,
                      rlons_present_subset, rlats_present_subset,
                      sub_cache, craton_cache, a, g):

    index_deposit_age = time_indices
    if len(index_deposit_age) == 0:
        return None

    rlons         = rlons_subset.copy()
    rlats         = rlats_subset.copy()
    rlons_present = rlons_present_subset.copy()
    rlats_present = rlats_present_subset.copy()

    valid_mask    = (rlats >= -90) & (rlats <= 90) & (rlons >= -180) & (rlons <= 180)
    rlons         = rlons[valid_mask]
    rlats         = rlats[valid_mask]
    rlons_present = rlons_present[valid_mask]
    rlats_present = rlats_present[valid_mask]

    if len(rlats) == 0:
        return None

    # Pull pre-computed data for formation time
    sc  = sub_cache[time]
    cc  = craton_cache[time]
    craton_centroids  = cc['centroids']
    tree_craton       = cc['tree']
    craton_geometries = cc['geometries']

    # ==================== INSTANTANEOUS VALUES ====================
    dep_vel     = model.get_point_velocities(rlons, rlats, time)
    dep_vel_mag = np.hypot(dep_vel[:, 0], dep_vel[:, 1])

    dist_to_craton, cr_neighbours = KD_dist(
        craton_centroids[:, 1], craton_centroids[:, 0], rlons, rlats
    )

    instantaneous_results = {}

    for loop_idx, (d2r, neighbour, df_index) in enumerate(
            zip(dist_to_craton, cr_neighbours, index_deposit_age)):

        if not valid_mask[loop_idx]:
            continue

        geom          = craton_geometries[neighbour]
        craton_coords = geom.to_lat_lon_array()

        yx            = craton_coords[1:] - craton_coords[:-1]
        craton_angles = np.rad2deg(np.arctan2(yx[:, 1], yx[:, 0]))
        craton_angles[craton_angles < 0] += 180

        cr_vel        = model.get_point_velocities(craton_coords[:, 1], craton_coords[:, 0], time)
        cr_vel_mag    = np.hypot(cr_vel[:, 0], cr_vel[:, 1])

        rx, ry, rz    = gplately.tools.lonlat2xyz(craton_coords[:, 1], craton_coords[:, 0])
        d2s, sz_nbrs  = sc['tree'].query(np.c_[rx, ry, rz])
        d2s_mean      = d2s.mean() * 6371

        sz_conv       = sc['conv'][sz_nbrs].mean()
        sz_migr       = sc['migr'][sz_nbrs].mean()
        sz_norm_v     = sc['norm'][sz_nbrs].copy()
        sz_norm_v[sz_norm_v > 180] -= 180

        obliquity     = abs(sz_norm_v.mean() - craton_angles.mean()) % 180
        if obliquity > 90:
            obliquity = 180 - obliquity

        instantaneous_results[df_index] = [
            cr_vel_mag.mean(),        # cr_ve
            dep_vel_mag[loop_idx],    # dp_ve
            d2s_mean,                 # ds_cr_tr
            sz_conv,                  # sb_ve
            sz_migr,                  # tr_adv
            obliquity,                # tr_cr_an
            rlats[loop_idx],          # lat
            rlons[loop_idx]           # lon
        ]

    # ==================== MEDIAN + GRADIENT (two time points only) ====================
    # t0 = time (formation), t1 = time - 20 (clamped to 0)
    t0 = time
    t1 = max(0, time - g)

    # --- positions at t0 (fully reconstructed) ---
    i_lons_t0, i_lats_t0 = rlons, rlats  # already at formation time

    # --- positions at t1 (linearly interpolated) ---
    i_lons_t1, i_lats_t1 = interpolate_positions(
        rlons, rlats, rlons_present, rlats_present, t1, time
    )

    # Pull cached subduction and craton for both time points
    sc_t0 = sub_cache[t0]
    sc_t1 = sub_cache[t1]
    cc_t0 = craton_cache[t0]
    cc_t1 = craton_cache[t1]

    dep_xyz_t0 = np.c_[gplately.tools.lonlat2xyz(i_lons_t0, i_lats_t0)]
    dep_xyz_t1 = np.c_[gplately.tools.lonlat2xyz(i_lons_t1, i_lats_t1)]

    # KDTree lookups
    _, sz_nbrs_t0 = sc_t0['tree'].query(dep_xyz_t0)
    _, sz_nbrs_t1 = sc_t1['tree'].query(dep_xyz_t1)
    _, cr_nbrs_t0 = cc_t0['tree'].query(dep_xyz_t0)
    _, cr_nbrs_t1 = cc_t1['tree'].query(dep_xyz_t1)

    # Deposit velocities at both time points
    dv_t0   = model.get_point_velocities(i_lons_t0, i_lats_t0, t0)
    dv_mag_t0 = np.hypot(dv_t0[:, 0], dv_t0[:, 1])

    dv_t1   = model.get_point_velocities(i_lons_t1, i_lats_t1, t1)
    dv_mag_t1 = np.hypot(dv_t1[:, 0], dv_t1[:, 1])

    # Craton velocities at both time points
    cv_t0   = model.get_point_velocities(
        cc_t0['centroids'][cr_nbrs_t0, 1], cc_t0['centroids'][cr_nbrs_t0, 0], t0
    )
    cv_mag_t0 = np.hypot(cv_t0[:, 0], cv_t0[:, 1])
    cv_az_t0  = np.rad2deg(np.arctan2(cv_t0[:, 0], cv_t0[:, 1]))
    cv_az_t0[cv_az_t0 < 0] += 180

    cv_t1   = model.get_point_velocities(
        cc_t1['centroids'][cr_nbrs_t1, 1], cc_t1['centroids'][cr_nbrs_t1, 0], t1
    )
    cv_mag_t1 = np.hypot(cv_t1[:, 0], cv_t1[:, 1])
    cv_az_t1  = np.rad2deg(np.arctan2(cv_t1[:, 0], cv_t1[:, 1]))
    cv_az_t1[cv_az_t1 < 0] += 180

    # Subduction properties at both time points
    conv_t0 = sc_t0['conv'][sz_nbrs_t0].mean()
    migr_t0 = sc_t0['migr'][sz_nbrs_t0].mean()
    norm_t0 = sc_t0['norm'][sz_nbrs_t0].copy()
    norm_t0[norm_t0 > 180] -= 180

    conv_t1 = sc_t1['conv'][sz_nbrs_t1].mean()
    migr_t1 = sc_t1['migr'][sz_nbrs_t1].mean()
    norm_t1 = sc_t1['norm'][sz_nbrs_t1].copy()
    norm_t1[norm_t1 > 180] -= 180

    # Obliquity at both time points
    obliq_t0 = abs(norm_t0.mean() - cv_az_t0.mean()) % 180
    if obliq_t0 > 90:
        obliq_t0 = 180 - obliq_t0

    obliq_t1 = abs(norm_t1.mean() - cv_az_t1.mean()) % 180
    if obliq_t1 > 90:
        obliq_t1 = 180 - obliq_t1

    # Median = mean of the two time points
    # Gradient = t1 value - t0 value  (change over the 20 Ma window)
    median_gradient_data = [
        np.mean([dv_mag_t0.mean(),  dv_mag_t1.mean()]),  # dp_ve_m
        np.mean([cv_mag_t0.mean(),  cv_mag_t1.mean()]),  # cr_ve_m
        np.mean([cv_az_t0.mean(),   cv_az_t1.mean()]),   # cr_ve_az_m
        np.mean([obliq_t0,          obliq_t1]),           # tr_cr_an_m
        np.mean([conv_t0,           conv_t1]),            # sb_ve_m
        np.mean([migr_t0,           migr_t1]),            # tr_adv_m
        dv_mag_t1.mean()  - dv_mag_t0.mean(),            # dp_ve_g
        cv_mag_t1.mean()  - cv_mag_t0.mean(),            # cr_ve_g
        cv_az_t1.mean()   - cv_az_t0.mean(),             # cr_ve_az_c
        obliq_t1          - obliq_t0,                    # tr_cr_an_c
        conv_t1           - conv_t0,                     # sb_ve_g
        migr_t1           - migr_t0                      # tr_adv_g
    ]

    return {
        'time':            time,
        'instantaneous':   instantaneous_results,
        'median_gradient': median_gradient_data,
        'indices':         index_deposit_age.tolist()
    }


# ==================== STEP 6: Run Parallel Processing ====================
print(f"\nProcessing {len(unique_times)} time steps in parallel on 20 cores...")

parallel_tasks = []
for time in unique_times:
    indices = time_to_indices[time]
    parallel_tasks.append((
        time, indices,
        pts_lons[indices],         pts_lats[indices],
        pts_lons_present[indices], pts_lats_present[indices]
    ))

print(f"Prepared {len(parallel_tasks)} tasks")

all_results = Parallel(n_jobs=20, verbose=10)(
    delayed(process_time_step)(
        time, indices, lons, lats, lons_p, lats_p,
        sub_cache, craton_cache, a, g
    )
    for time, indices, lons, lats, lons_p, lats_p in parallel_tasks
)

# ==================== STEP 7: FAST MERGE RESULTS ====================
print("\n" + "="*80)
print("FAST MERGE: Building data structures...")
print("="*80)

result_columns = [
    'cr_ve', 'dp_ve', 'ds_cr_tr', 'sb_ve', 'tr_adv', 'tr_cr_an', 'lat', 'lon',
    'dp_ve_m', 'cr_ve_m', 'cr_ve_az_m', 'tr_cr_an_m', 'sb_ve_m', 'tr_adv_m',
    'dp_ve_g', 'cr_ve_g', 'cr_ve_az_c', 'tr_cr_an_c', 'sb_ve_g', 'tr_adv_g', 'ds_dp_tr'
]
for col in result_columns:
    if col not in df.columns:
        df[col] = np.nan

instantaneous_cols   = ['cr_ve', 'dp_ve', 'ds_cr_tr', 'sb_ve', 'tr_adv', 'tr_cr_an', 'lat', 'lon']
median_gradient_cols = [
    'dp_ve_m', 'cr_ve_m', 'cr_ve_az_m', 'tr_cr_an_m', 'sb_ve_m', 'tr_adv_m',
    'dp_ve_g', 'cr_ve_g', 'cr_ve_az_c', 'tr_cr_an_c', 'sb_ve_g', 'tr_adv_g'
]

inst_data = {col: {} for col in instantaneous_cols}
mg_data   = {col: {} for col in median_gradient_cols}

for result_idx, result in enumerate(all_results):
    if result is None:
        continue
    if result_idx % 100 == 0:
        print(f"  Merging result {result_idx}/{len(all_results)}...")

    for idx, values in result['instantaneous'].items():
        for ci, col in enumerate(instantaneous_cols):
            inst_data[col][idx] = values[ci]

    if result['median_gradient'] is not None:
        for idx in result['indices']:
            for ci, col in enumerate(median_gradient_cols):
                mg_data[col][idx] = result['median_gradient'][ci]

print("Updating DataFrame (bulk)...")
for col in instantaneous_cols:
    if inst_data[col]:
        df.loc[list(inst_data[col].keys()), col] = list(inst_data[col].values())
        print(f"  {col}: {len(inst_data[col])} values")

for col in median_gradient_cols:
    if mg_data[col]:
        df.loc[list(mg_data[col].keys()), col] = list(mg_data[col].values())
        print(f"  {col}: {len(mg_data[col])} values")

print("\nDone!")

Reading CSV file...
Initial rows: 18010
Filtered rows: 18000
Coordinates extracted:
  - Reconstructed lon range: [-179.99, 180.00]
  - Reconstructed lat range: [-87.63, 87.07]
  - Total points: 18000

Found 1800 unique time steps

Pre-computing tessellations for 1801 time steps...
(Only 2 time points per deposit time: t and t-20)
  Tessellating subduction zones: 0/1801 (time=0 Ma)...
  Tessellating subduction zones: 100/1801 (time=100 Ma)...
  Tessellating subduction zones: 200/1801 (time=200 Ma)...
  Tessellating subduction zones: 300/1801 (time=300 Ma)...
  Tessellating subduction zones: 400/1801 (time=400 Ma)...
  Tessellating subduction zones: 500/1801 (time=500 Ma)...
  Tessellating subduction zones: 600/1801 (time=600 Ma)...
  Tessellating subduction zones: 700/1801 (time=700 Ma)...
  Tessellating subduction zones: 800/1801 (time=800 Ma)...
  Tessellating subduction zones: 900/1801 (time=900 Ma)...
  Tessellating subduction zones: 1000/1801 (time=1000 Ma)...
  Tessellating subduc

[Parallel(n_jobs=20)]: Using backend LokyBackend with 20 concurrent workers.
[Parallel(n_jobs=20)]: Done   1 tasks      | elapsed:  2.4min
C:\Users\hshi9735\AppData\Local\anaconda3\envs\STAMP\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
[Parallel(n_jobs=20)]: Done  10 tasks      | elapsed: 12.7min
[Parallel(n_jobs=20)]: Done  21 tasks      | elapsed: 25.4min
[Parallel(n_jobs=20)]: Done  32 tasks      | elapsed: 37.6min
[Parallel(n_jobs=20)]: Done  45 tasks      | elapsed: 52.6min
[Parallel(n_jobs=20)]: Done  58 tasks      | elapsed: 66.9min
[Parallel(n_jobs=20)]: Done  73 tasks      | elapsed: 84.0min
[Parallel(n_jobs=20)]: Done  88 tasks      | elapsed: 100.8min
[Parallel(n_jobs=20)]: Done 105 tasks      | elapsed: 119.9min
[Parallel(n_jobs=20)]: Done 122 tasks      | elapsed: 139.3min
[Parallel(n_jobs=20)]: Done 141 tasks 


FAST MERGE: Building data structures...
  Merging result 0/1800...
  Merging result 100/1800...
  Merging result 200/1800...
  Merging result 300/1800...
  Merging result 400/1800...
  Merging result 500/1800...
  Merging result 600/1800...
  Merging result 700/1800...
  Merging result 800/1800...
  Merging result 900/1800...
  Merging result 1000/1800...
  Merging result 1100/1800...
  Merging result 1200/1800...
  Merging result 1300/1800...
  Merging result 1400/1800...
  Merging result 1500/1800...
  Merging result 1600/1800...
  Merging result 1700/1800...
Updating DataFrame (bulk)...
  cr_ve: 18000 values
  dp_ve: 18000 values
  ds_cr_tr: 18000 values
  sb_ve: 18000 values
  tr_adv: 18000 values
  tr_cr_an: 18000 values
  lat: 18000 values
  lon: 18000 values
  dp_ve_m: 18000 values
  cr_ve_m: 18000 values
  cr_ve_az_m: 18000 values
  tr_cr_an_m: 18000 values
  sb_ve_m: 18000 values
  tr_adv_m: 18000 values
  dp_ve_g: 18000 values
  cr_ve_g: 18000 values
  cr_ve_az_c: 18000 valu

In [28]:
# # ==================== STEP 6: Calculate Final Distances to Trench ====================
# print("\n" + "="*80)
# print("Calculating final distances to trench...")
# print("="*80)

# unique_times_sorted = sorted(unique_times)

# # Process final distances in chunks to avoid memory issues
# chunk_size = 100000
# num_chunks = (len(df) + chunk_size - 1) // chunk_size

# print(f"Processing final distances in {num_chunks} chunks...")

# # Tessellate subduction zones for the last time step ONCE
# subduction_data_final = model.tessellate_subduction_zones(unique_times_sorted[-1], np.deg2rad(0.5), ignore_warnings=True)
# subduction_lon_final = subduction_data_final[:, 0]
# subduction_lat_final = subduction_data_final[:, 1]

# sx_final, sy_final, sz_final = gplately.tools.lonlat2xyz(subduction_lon_final, subduction_lat_final)
# tree_sz_final = cKDTree(np.c_[sx_final, sy_final, sz_final])

# for chunk_idx in range(num_chunks):
#     start_idx = chunk_idx * chunk_size
#     end_idx = min((chunk_idx + 1) * chunk_size, len(df))
    
#     if chunk_idx % 5 == 0:
#         print(f"  Processing chunk {chunk_idx+1}/{num_chunks}...")
    
#     # Create Points object for this chunk
#     pts_chunk = gplately.Points(model, pts_lons[start_idx:end_idx], pts_lats[start_idx:end_idx])
    
#     # Reconstruct points for final time
#     #rlons_final, rlats_final = pts_chunk.reconstruct(unique_times_sorted[-1], return_array=True)
#     rlons_final = df['lon'].values
#     rlats_final = df['lat'].values
    
    
#     # Handle length mismatch - only process what we got back
#     actual_returned = len(rlons_final)
#     expected_length = end_idx - start_idx
    
#     if actual_returned != expected_length:
#         print(f"    Warning: chunk {chunk_idx+1} expected {expected_length} points but got {actual_returned}")
#         # Adjust end_idx to match what we actually got
#         end_idx = start_idx + actual_returned
    
#     valid_mask_final = (rlats_final >= -90) & (rlats_final <= 90) & (rlons_final >= -180) & (rlons_final <= 180)
#     rlons_final_valid = rlons_final[valid_mask_final]
#     rlats_final_valid = rlats_final[valid_mask_final]
    
#     if len(rlons_final_valid) > 0:
#         dx_final, dy_final, dz_final = gplately.tools.lonlat2xyz(rlons_final_valid, rlats_final_valid)
#         d2s_final, _ = tree_sz_final.query(np.c_[dx_final, dy_final, dz_final])
#         d2s_final = d2s_final * 6371
        
#         # Create indices array that matches the actual returned length
#         chunk_indices = np.arange(start_idx, end_idx)
#         valid_indices = chunk_indices[valid_mask_final]
        
#         # Assign values to the valid indices
#         df.loc[valid_indices, 'ds_dp_tr'] = d2s_final
    
#     del pts_chunk
#     gc.collect()

# # # ==================== STEP 7: Save Results ====================
# # print("\n" + "="*80)
# # print("Saving results...")
# # df.to_csv('./Outputs/backarc_data_0.5.csv', index=False)
# # print("Analysis complete! Results saved to ./Outputs/backarc_data_0.5.csv")
# # print("="*80)

In [29]:
df['lat'] = df['lat'].abs()

In [30]:
# ==================== STEP 6: Calculate Final Distances to Trench ====================
print("\n" + "="*80)
print("Calculating final distances to trench...")
print("="*80)

unique_times_sorted = sorted(unique_times)

# Tessellate subduction zones for the last time step ONCE
subduction_data_final = model.tessellate_subduction_zones(unique_times_sorted[-1], np.deg2rad(0.5), ignore_warnings=True)
subduction_lon_final = subduction_data_final[:, 0]
subduction_lat_final = subduction_data_final[:, 1]

sx_final, sy_final, sz_final = gplately.tools.lonlat2xyz(subduction_lon_final, subduction_lat_final)
tree_sz_final = cKDTree(np.c_[sx_final, sy_final, sz_final])

# Use lon/lat directly — already reconstructed, no chunking needed
rlons_final = df['lon'].values
rlats_final = df['lat'].values

valid_mask_final = (rlats_final >= -90) & (rlats_final <= 90) & (rlons_final >= -180) & (rlons_final <= 180)
rlons_final_valid = rlons_final[valid_mask_final]
rlats_final_valid = rlats_final[valid_mask_final]

print(f"  Valid points: {valid_mask_final.sum()} / {len(df)}")

if len(rlons_final_valid) > 0:
    dx_final, dy_final, dz_final = gplately.tools.lonlat2xyz(rlons_final_valid, rlats_final_valid)
    d2s_final, _ = tree_sz_final.query(np.c_[dx_final, dy_final, dz_final])
    d2s_final = d2s_final * 6371

    valid_indices = np.where(valid_mask_final)[0]
    df.loc[valid_indices, 'ds_dp_tr'] = d2s_final
    print(f"  Assigned ds_dp_tr to {len(valid_indices)} rows")

print("Final distances calculated successfully!")

# ==================== STEP 7: Save Results ====================
print("\n" + "="*80)
print("Saving results...")
df.to_csv('./Outputs/unlabelled_data.csv', index=False)
print("Analysis complete! Results saved to ./Outputs/unlabelled_data.csv")
print("="*80)


Calculating final distances to trench...
  Valid points: 18000 / 18000
  Assigned ds_dp_tr to 18000 rows
Final distances calculated successfully!

Saving results...
Analysis complete! Results saved to ./Outputs/unlabelled_data.csv


# cumulative_subducted_water_mantle

# cumulative_subducted_carbon_crust

In [ ]:
# # ==================== STEP 6.5: Extract Raster Values from NetCDF ====================
# print("\n" + "="*80)
# print("Extracting raster values from NetCDF files...")
# print("="*80)

# import os
# from netCDF4 import Dataset
# from scipy.interpolate import RegularGridInterpolator

# # ========== FIRST EXTRACTION: Mantle Subducted Water ==========
# print("\n[1/2] Processing Mantle Subducted Water...")
# raster_folder_water = r"C:/Users/Hojat/OneDrive - The University of Sydney (Students)/PhD_3_Notebooks/Mantle_Water/mantle_subducted_water_500km/"

# def get_raster_value_water(age, lon, lat):
#     raster_file = f"cumulative_subducted_water_mantle_{int(age)}.nc"
#     raster_path = os.path.join(raster_folder_water, raster_file)
    
#     if not os.path.exists(raster_path):
#         return np.nan
    
#     try:
#         with Dataset(raster_path, 'r') as f:
#             lon_grid = f.variables['lon'][:]
#             lat_grid = f.variables['lat'][:]
#             values = f.variables['z'][:]
            
#             interpolator = RegularGridInterpolator(
#                 (lat_grid, lon_grid), 
#                 values, 
#                 bounds_error=False, 
#                 fill_value=np.nan
#             )
#             point = np.array([[lat, lon]])
#             return interpolator(point)[0]
#     except Exception as e:
#         print(f"  Error processing water age {age} at ({lon}, {lat}): {e}")
#         return np.nan

# # Process in chunks
# print(f"Processing {len(df)} rows...")
# chunk_size = 100000
# num_chunks = (len(df) + chunk_size - 1) // chunk_size

# df['CSWM'] = np.nan

# for chunk_idx in range(num_chunks):
#     start_idx = chunk_idx * chunk_size
#     end_idx = min((chunk_idx + 1) * chunk_size, len(df))
    
#     if chunk_idx % 5 == 0:
#         print(f"  Processing chunk {chunk_idx+1}/{num_chunks} (rows {start_idx} to {end_idx})...")
    
#     df.loc[start_idx:end_idx-1, 'CSWM'] = df.loc[start_idx:end_idx-1].apply(
#         lambda row: get_raster_value_water(row['age (Ma)'], row['lon'], row['lat']), 
#         axis=1
#     )

# print("Mantle water extraction complete!")
# print(f"  - Non-NaN values: {df['CSWM'].notna().sum()}")
# print(f"  - NaN values: {df['CSWM'].isna().sum()}")

# # ========== SECOND EXTRACTION: Crustal Carbon ==========
# print("\n[2/2] Processing Crustal Carbon...")
# raster_folder_carbon = r"C:/Users/Hojat/OneDrive - The University of Sydney (Students)/PhD_3_Notebooks/Crustal_Carbon/"

# def get_raster_value_carbon(age, lon, lat):
#     raster_file = f"cumulative_subducted_carbon_crust_{int(age)}.nc"
#     raster_path = os.path.join(raster_folder_carbon, raster_file)
    
#     if not os.path.exists(raster_path):
#         return np.nan
    
#     try:
#         with Dataset(raster_path, 'r') as f:
#             lon_grid = f.variables['lon'][:]
#             lat_grid = f.variables['lat'][:]
#             values = f.variables['z'][:]
            
#             interpolator = RegularGridInterpolator(
#                 (lat_grid, lon_grid), 
#                 values, 
#                 bounds_error=False, 
#                 fill_value=np.nan
#             )
#             point = np.array([[lat, lon]])
#             return interpolator(point)[0]
#     except Exception as e:
#         print(f"  Error processing carbon age {age} at ({lon}, {lat}): {e}")
#         return np.nan

# df['CSCC'] = np.nan

# for chunk_idx in range(num_chunks):
#     start_idx = chunk_idx * chunk_size
#     end_idx = min((chunk_idx + 1) * chunk_size, len(df))
    
#     if chunk_idx % 5 == 0:
#         print(f"  Processing chunk {chunk_idx+1}/{num_chunks} (rows {start_idx} to {end_idx})...")
    
#     df.loc[start_idx:end_idx-1, 'CSCC'] = df.loc[start_idx:end_idx-1].apply(
#         lambda row: get_raster_value_carbon(row['age (Ma)'], row['lon'], row['lat']), 
#         axis=1
#     )

# print("Crustal carbon extraction complete!")
# print(f"  - Non-NaN values: {df['CSCC'].notna().sum()}")
# print(f"  - NaN values: {df['CSCC'].isna().sum()}")

# print("\nAll raster extractions complete!")

# # ==================== STEP 7: Save Results ====================
# print("\n" + "="*80)
# print("Saving results...")
# df.to_csv('./Outputs/coastline_data_0.5.csv', index=False)
# print("Analysis complete! Results saved to ./Outputs/coastline_data_0.5.csv")
# print("="*80)

In [ ]:
df.to_csv('./Outputs/coastline_data_no_recon.csv', index=False)
print("Analysis complete! Results saved to ./Outputs/coastline_data_no_recon.csv")

In [ ]:
# # ==================== Option 2: Display from current dataframe ====================
# print("\n" + "="*80)
# print("Displaying first 10 rows of current dataframe...")
# print("="*80)

# print(f"\nTotal rows: {len(df)}")
# print(f"Total columns: {len(df.columns)}")
# print(f"\nColumn names:")
# print(df.columns.tolist())
# print("\n" + "="*80)
# print("First 10 rows:")
# print("="*80)
# print(df.head(10))

# # Optional: Display with better formatting
# print("\n" + "="*80)
# print("First 10 rows (transposed for better readability):")
# print("="*80)
# print(df.head(10).T)

In [ ]:
# # ==================== Option 1: Read from saved CSV file ====================
# print("\n" + "="*80)
# print("Reading saved CSV file and displaying first 10 rows...")
# print("="*80)

# saved_df = pd.read_csv('./Outputs/unlabelled_data.csv')
# print(f"\nTotal rows in saved file: {len(saved_df)}")
# print(f"Total columns: {len(saved_df.columns)}")
# print(f"\nColumn names:")
# print(saved_df.columns.tolist())
# print("\n" + "="*80)
# print("First 10 rows:")
# print("="*80)
# print(saved_df.head(10))

# # Optional: Display with better formatting
# print("\n" + "="*80)
# print("First 10 rows (transposed for better readability):")
# print("="*80)
# print(saved_df.head(10).T)